In [167]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [168]:
df = pd.read_csv("/content/drive/MyDrive/Academic_Project/DataSet/data.csv")

In [169]:
# for col in df.columns:
#     print(col)

In [170]:
print(df.shape)
df = df.dropna(how="all")   # remove last row
df = df.reset_index(drop=True)
print(df.shape)

(501, 63)
(500, 63)


In [171]:
# Full column rename mapping

rename_map = {
    "Do you work as well as Study?(পড়াশোনার পাশাপাশি চাকরি করেন?)":
        "Work_While_Study",

    "Residential Area (আবাসিক এলাকা)":
        "Residential_Area",

    "Social Economic Status (আর্থ-সামাজিক অবস্থা)":
        "Social Economic Status",

    "Do you feel any financial pressure?(আপনি কি কোনো আর্থিক চাপ অনুভব করছেন?)":
        "Financial_Pressure",

    "Does the participant have any debts?(অংশগ্রহণকারী কি কোন ঋণ আছে?)":
        "Has_Debts",

    "Are you satisfied with your current living environment? (আপনি কি আপনার বর্তমান বসবাসের পরিবেশে সন্তুষ্ট?)":
        "Satisfied_Living_Environment",

    "Have you recently lost someone close to you? (আপনি সম্প্রতি আপনার কাছের কাউকে হারিয়েছেন কি না )":
        "Lost_Someone_Recently",

    "You are actively engaging as a participant in physical exertion.(আপনি সক্রিয়ভাবে শারীরিক পরিশ্রমে অংশগ্রহণকারী হিসাবে জড়িত।)":
        "Physical_Activity",

    'Are you afflicted by any significant ailments?(আপনি কি কোনও গুরুতর অসুখে ভুগছেন?")':
        "Significant_Ailments",

    "Are you currently on any prescribed medication?(আপনি কি বর্তমানে কোনও ওষুধ সেবন করছেন?)":
        "On_Medication",

    "Are you accustomed to smoking?(আপনি কি ধূমপানে অভ্যস্ত?)":
        "Smoking",

    "Do you consume alcoholic beverages?(আপনি কি অ্যালকোহলযুক্ত পানীয় গ্রহণ করেন?)":
        "Alcohol_Consumption",

    "What is your average nightly sleep duration in hours?(আপনার রাতের ঘুমের গড় সময়কাল কত ঘন্টার মধ্যে?)":
        "Sleep_Duration",

    "Average hours that the participant spends in social network (in a day)(অংশগ্রহণকারী সামাজিক নেটওয়ার্কে ব্যয় করে এমন গড় ঘন্টা (এক দিনে))":
        "Social_Media_Hours",

    "Do you have current workload or academic demands?(আপনার কি বর্তমান কাজের চাপ বা একাডেমিক চাহিদা রয়েছে?)":
        "Workload_Academic_Demand",

    "Presently, I feel melancholic. (ইদানিং আমি মনমরা থাকি। )":
        "Melancholic",

    "My future appears shrouded in darkness. (আমার ভবিষ্যৎ অন্ধকার ম‌নে হয়।)":
        "Future_Hopelessness",

    "At present, I perceive myself as utterly unsuccessful as a human being. (বর্তমানে আমি অনুভব করি যে মানুষ হিসেবে আমি সম্পূর্ণ ব্যর্থ।)":
        "Self_Perceived_Failure",

    "I find no interest in anything whatsoever. (আমি কোন কিছুতেই আগ্রহ পাই না। )":
        "Interest_Loss",

    "Life feels devoid of meaning.(জীবনটা অর্থহীন।)":
        "Meaninglessness",

    "I feel like, All has come to an end for me. (আমার সব শেষ হয়ে গেছে, এমনটা ম‌নে হয়।)":
        "Hopelessness_EndFeeling",

    "I feel remarkably insignificant within myself. (নিজেকে খুব ছোট মনে হয় আমার।)":
        "Feeling_Insignificant",

    "Everything seems to have eroded my self-confidence.(সব কিছুতে আমার আত্মবিশ্বাস কমে গেছে।)":
        "Self_Confidence_Erosion",

    "Have you recently entertained any suicidal or self-harming thoughts?(আপনি ইদানীং কোনো আত্মঘাতী বা আত্ম-ক্ষতির চিন্তাভাবনা উপভোগ করেছেন কি না ?)":
        "Suicidal_Thoughts",

    "I often get tears. (প্রায়ই আমার কান্না পায়।)":
        "Crying_Frequency",

    "I'm easily agitated. (আমি সহজেই উত্তেজিত। )":
        "Agitation_Level",

    "I can't participate in social activities like I used to.(সামাজিক কাজকর্মে আগের মতো অংশগ্রহন করতে পারি না।)":
        "Social_Withdrawal",

    "I suffer from indecisiveness. (আমি সিদ্ধান্তহীনতায় ভুগি।)":
        "Indecisiveness",

    "I find myself devoid of joy anywhere. (আমি কোথাও আনন্দ ফুর্তি পাই না।)":
        "Anhedonia_No_Joy",

    "How often do you feel weak and fatigued easily?(আপনি কত ঘন ঘন দুর্বল এবং সহজেই ক্লান্ত বোধ করেন?)":
        "Fatigue_Frequency",

    """Insomnia (difficulty falling asleep, broken sleep, unsatisfying sleep, fatigue on waking, dreams, nightmares, and night terrors.)
অনিদ্রা (ঘুমিয়ে পড়তে অসুবিধা, ভাঙা ঘুম, অতৃপ্ত ঘুম, জেগে ওঠার ক্লান্তি, স্বপ্ন, দুঃস্বপ্ন এবং রাতের আতঙ্ক।""":
    "Insomnia",

    "My temper is irritable. (আমার মেজাজ খিঁটখিঁটে হয়ে গেছে। )":
        "Irritability",

    "My appetite has diminished. (আমার ক্ষুধা কমে গেছে।)":
        "Low_Appetite",

    "Concentrating on one topic is quite taxing for me.(একটি বিষয়ে মনোনিবেশ করা আমার জন্য বেশ করকর।)":
        "Difficulty_Focusing",

    "I feel weak and fatigued easily.(আমি সহজেই দুর্বল এবং ক্লান্ত বোধ করি।)":
        "Easy_Fatigue",

    "Lately, I find it challenging to focus on many things. (আমি আজকাল অনেক কিছুতে মনোযোগ দিতে পারি না।)":
        "Low_Concentration",

    "I find it difficult to speak in my social environment. (আমার সামাজিক পরিবেশে কথা বলতে অসুবিধা হয়।)":
        "Difficulty_Speaking_Socially",

    "My appetite has increased. (আমার ক্ষুধা বেড়ে গেছে।)":
        "High_Appetite",

    " I'm plagued by a sense of unrest. (আমি অস্থিরতার অনুভূতিতে জর্জরিত।)/(আমার অশান্তি লাগে। )":
        "Restlessness",

    "I reckon life has become exceedingly arduous presently. (আমি মনে করি যে,জীনবটা বর্তমানে খুব বেশী কষ্টকর।)":
        "Life_Feels_Hard",

    "I'm afraid something very bad will happen.(আমার খুব খারাপ কিছু ঘটবে বলে আশংকা হয়।)":
        "Fear_Something_Bad",

    "Whether the participant has recently felt abused (physically, emotionally, Sexual Harassment) or Not (অংশগ্রহণকারী সম্প্রতি নির্যাতিত (শারীরিক, মানসিক, যৌন হয়রানি) অনুভব করেছেন কি না)":
        "Recent_Abuse_Experience",

    "I feel that people show me compassion. (আমার মনে হয় মানুষ আমাকে করুণা করে।)":
        "Feels_Pitied",

    "I find myself absence of pleasure everywhere.  (আমি সব জায়গায় নিজেকে আনন্দের অনুপস্থিতি খুঁজে পাই।)":
        "Lack_of_Pleasure",

    "Presently, I'm feeling down.  (বর্তমানে, আমি খারাপ বোধ করছি।)":
        "Feeling_Down",

    "I feel that people are kind toward me. (আমি অনুভব করি যে লোকেরা আমার প্রতি সদয়।)":
        "Feels_Others_Are_Kind",

    "I find myself unable to perform educational and professional duties as before. (শিক্ষা ও পেশাগত কাজকর্ম আগের মতো করতে পারি না।)":
        "Performance_Decline",

    "How often do you feel like you don't have anyone to share your feelings with? (আপনি কতবার মনে করেন যে আপনার সাথে আপনার অনুভূতি ভাগ করার মতো কেউ নেই?)":
        "Share_Feelings_Lack",

    "How much do you feel left out in social situations? (সামাজিক পরিস্থিতিতে আপনি কতটা বাকী বোধ করেন?)":
        "Social_LeftOut_Level",

    "How often do you feel isolated from others?(আপনি কতবার অন্যদের থেকে বিচ্ছিন্ন বোধ করেন?)":
        "Isolation_Frequency",

    "How frequently do you feel there’s no one you can rely on for support?(আপনি কত ঘন ঘন মনে করেন যে সমর্থনের জন্য আপনি নির্ভর করতে পারেন এমন কেউ নেই?)":
        "No_Support_Frequency",

    "How frequently do you experience feelings of loneliness? (আপনি কত ঘন ঘন একাকীত্বের অনুভূতি অনুভব করেন?)":
        "Loneliness_Frequency",

    "How frequently do you feel aligned with the emotions or thoughts of the people around you? (আপনি কত ঘন ঘন আপনার চারপাশের লোকেদের আবেগ বা চিন্তার সাথে একত্রিত বোধ করেন?)":
        "Emotional_Alignment_Frequency",

    "How frequently do you perceive that people are around you but not genuinely present with you? (আপনি কত ঘন ঘন বুঝতে পারেন যে লোকেরা আপনার চারপাশে আছে কিন্তু সত্যিকারের আপনার সাথে উপস্থিত নয়?)":
        "Presence_Not_Genuine_Frequency",

    "To what extent do you experience your relationships with others as being unimportant?(অন্যদের সাথে আপনার সম্পর্ককে গুরুত্বহীন বলে আপনি কতটা অনুভব করেন?)":
        "Relationships_Unimportant_Level",
}

df = df.rename(columns=rename_map)

In [172]:
# Relationship column cleaning
dummies = pd.get_dummies(df["Relationship Status"], prefix="Relationship_Status").astype(int)
df = pd.concat([df.drop(columns=["Relationship Status"]), dummies], axis=1)

In [173]:
# Academic year ordinal encoding
mapping = {
    "1 Year": 1,
    "1 Year ": 1,
    "2 Year ": 2,
    "2 Year": 2,
    "3 Year": 3,
    "4 Year": 4
}
df["Academic Status"] = df["Academic Status"].map(mapping)

In [174]:
# Social Economic Status Ordinal encoding
mapping = {
    "Upper": 5,

    "Uper-Middle": 4,
    "Upper-Middle": 4,

    "Middle": 3,

    "Lower-Middle": 2,
    "Lowe-Middle": 2,

    "Lower": 1,
    "lower": 1
}
df["Social Economic Status"] = (
    df["Social Economic Status"].map(mapping)
)

In [175]:
# Gender column cleaning
df["Age"] = df["Age"].astype(int)
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})
yes_no_cols = ["Financial_Pressure", "Has_Debts", "Satisfied_Living_Environment",
               "Lost_Someone_Recently", "Physical_Activity", "Significant_Ailments",
               "On_Medication", "Smoking", "Alcohol_Consumption", "Workload_Academic_Demand",
               "Work_While_Study"
]
df[yes_no_cols] = df[yes_no_cols].replace({"Yes": 1, "No": 0})



/tmp/ipykernel_518/3900274640.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[yes_no_cols] = df[yes_no_cols].replace({"Yes": 1, "No": 0})


In [176]:
dummies = pd.get_dummies(df["Residential_Area"], prefix="Residential_Area").astype(int)
df = pd.concat([df.drop(columns=["Residential_Area"]), dummies], axis=1)

In [177]:
df["Sleep_Duration"] = df["Sleep_Duration"].str.strip().str.title()

df["Sleep_Duration"] = df["Sleep_Duration"].replace({
    "Low 5 Hours": "Below 5 Hours",
    "More Than 8 Hours": "More than 8 hours"  # unify casing
})

# 3. Map to numeric hour estimate
sleep_map = {
    "Below 5 Hours": 4,
    "5 Hours": 5,
    "6 Hours": 6,
    "7 Hours": 7,
    "8 Hours": 8,
    "More than 8 hours": 9
}

df["Sleep_Duration"] = df["Sleep_Duration"].map(sleep_map)

In [178]:
df["Social_Media_Hours"] = (
    df["Social_Media_Hours"]
    .astype(str)
    .str.strip()
    .str.replace("–", "-", regex=False)  # Replace special dash
    .str.title()
)

# Fix inconsistent formatting
df["Social_Media_Hours"] = df["Social_Media_Hours"].replace({
    "More Than 10 Hours A Day": "More than 10 hours a day",
    "Less Than 2 Hours": "Less than 2 hours"
})

# Numeric encoding (midpoint of ranges)
social_map = {
    "Less than 2 hours": 1,
    "2-4 Hours A Day": 3,
    "5-7 Hours A Day": 6,
    "8-10 Hours A Day": 9,
    "More than 10 hours a day": 11
}

df["Social_Media_Hours"] = df["Social_Media_Hours"].map(social_map)

In [179]:
melancholic_map = {
    "I do not feel melancholic.": 0,
    "I feel melancholic.": 1,
    "I feel melancholic most of the time, and I can't snap out of it.": 2,
    "I feel so deeply melancholic and unhappy that I can't stand it.": 3
}
df["Melancholic"] = df["Melancholic"].map(melancholic_map).astype(int)

In [180]:
future_map = {
    "I am not discouraged about my future.": 0,
    "I sometimes feel discouraged about my future.": 1,
    "I am moderately discouraged about my future.": 2,
    "I feel my future is hopeless and will only get worse.": 3
}
df["Future_Hopelessness"] = df["Future_Hopelessness"].map(future_map).astype(int)

In [181]:
failure_map = {
    "I do not feel like a failure.": 0,
    "I feel I have somewhat failed.": 1,
    "I feel I have failed more than I could have.": 2,
    "I feel I have mostly failed.": 3
}
df["Self_Perceived_Failure"] = df["Self_Perceived_Failure"].map(failure_map).astype(int)

In [182]:
interest_map = {
    "I have as much interest as I ever did in the things I enjoy.": 0,
    "I have a little less interest than I used to in the things I enjoy.": 1,
    "I have much less interest than I used to in the things I enjoy.": 2,
    "I have no interest in the things I used to enjoy.": 3
}
df["Interest_Loss"] = df["Interest_Loss"].map(interest_map).astype(int)

In [183]:
meaning_map = {
    "Life feels devoid of meaning none of the time.": 0,
    "Life feels devoid of meaning a good part of the time.": 1,
    "Life feels devoid of meaning most of the time.": 2,
    "Life feels devoid of meaning all of the time.": 3
}
df["Meaninglessness"] = df["Meaninglessness"].map(meaning_map).astype(int)

In [184]:
end_map = {
    "I don't feel like all has come to an end for me.": 0,
    "I feel like all may be coming to an end for me.": 1,
    "I expect that all will come to an end for me.": 2,
    "I feel like all has already come to an end for me.": 3
}
df["Hopelessness_EndFeeling"] = df["Hopelessness_EndFeeling"].map(end_map).astype(int)

In [185]:
insignificant_map = {
    "I don't feel insignificant in my own eyes.": 0,
    "I feel somewhat insignificant in my own eyes.": 1,
    "I feel overwhelmingly insignificant in my own eyes.": 2,
    "I feel deeply insignificant in my own eyes.": 3
}
df["Feeling_Insignificant"] = df["Feeling_Insignificant"].map(insignificant_map).astype(int)

In [186]:
self_conf_map = {
    "I don't feel my self-confidence is undermined by anything.": 0,
    "I am self-critical about my weaknesses or mistakes.": 1,
    "I blame myself for everything that diminishes my self-confidence.": 2,
    "I constantly blame myself for my faults.": 3
}
df["Self_Confidence_Erosion"] = df["Self_Confidence_Erosion"].map(self_conf_map).astype(int)

In [187]:
suicidal_map = {
    "I haven't considered any actions that could harm myself.": 0,
    "I have considered actions that could harm myself, but I wouldn't act on them.": 1,
    "I wish to harm myself.": 2,
    "I would harm myself if I could.": 3
}
df["Suicidal_Thoughts"] = df["Suicidal_Thoughts"].map(suicidal_map).astype(int)

In [188]:
crying_map = {
    "I don't tear up any more than usual.": 0,
    "I tear up more often now than I used to.": 1,
    "I tear up all the time now.": 2,
    "I used to be able to tear up, but now I can't even though I want to.": 3
}
df["Crying_Frequency"] = df["Crying_Frequency"].map(crying_map).astype(int)

In [189]:
agitated_map = {
    "I am not any more easily agitated than I usually am.": 0,
    "I am slightly more easily agitated now than usual.": 1,
    "I am frequently easily agitated.": 2,
    "I feel constantly easily agitated.": 3
}
df["Agitation_Level"] = df["Agitation_Level"].map(agitated_map).astype(int)

In [190]:
df["Share_Feelings_Lack"] = df["Share_Feelings_Lack"].replace({"Sometimess": "Sometimes"})

In [191]:
cool_map = ["Share_Feelings_Lack",
"Social_LeftOut_Level",
"Isolation_Frequency",
"No_Support_Frequency",
"Loneliness_Frequency",
"Emotional_Alignment_Frequency",
"Presence_Not_Genuine_Frequency",
"Relationships_Unimportant_Level"
]
clean_map = {
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Often": 3
}
for item in cool_map:
  df[item] = df[item].map(clean_map).astype(int)

In [192]:
# Depression level indicator cleaning
bdi_map = {
    "Minimal Depression": 0,
    "Mild Depression": 1,
    "Moderate Depression": 2,
    "Severe Depression": 3
}
df["Depression Level (BDI-II) (20 items)"] = df["Depression Level (BDI-II) (20 items)"].map(bdi_map).astype(int)


updates = {
    55: "Severe Depression",
    301: "Severe Depression",
    324: "Severe Depression",
    435: "Moderate Severe Depression"
}

for row, value in updates.items():
    df.loc[row, "Depression Level (PHQ-9 items)"] = value
phq_map = {
    "Minimal Depression": 0,
    "Mild Depression": 1,
    "Moderate Depression": 2,
    "Moderate Severe Depression": 3,
    "Severe Depression": 4
}
df["Depression Level (PHQ-9 items)"] = df["Depression Level (PHQ-9 items)"].map(phq_map)

cesd_map = {
    "Minimal or no Depressive": 0,
    "Mild Depression": 1,
    "Moderate Depression": 2,
    "Severe Depression": 3
}
df["Depression Level (CES-D) (20 items)"] = df["Depression Level (CES-D) (20 items)"].map(cesd_map).astype(int)

loneliness_map = {
    "Low degree of loneliness": 0,
    "Moderate degree of loneliness": 1,
    "Moderately high degree of loneliness": 2,
    "High degree of loneliness": 3
}
df["Loneliness Level (UCLA-8 items)"] = df["Loneliness Level (UCLA-8 items)"].map(loneliness_map).astype(int)

In [193]:
performance_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Performance_Decline"] = df["Performance_Decline"].astype(str).str.strip()
df["Performance_Decline"] = df["Performance_Decline"].map(performance_map)


In [194]:
feels_kind_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Feels_Others_Are_Kind"] = df["Feels_Others_Are_Kind"].astype(str).str.strip()
df["Feels_Others_Are_Kind"] = df["Feels_Others_Are_Kind"].map(feels_kind_map)


In [195]:
feeling_down_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Feeling_Down"] = df["Feeling_Down"].astype(str).str.strip()
df["Feeling_Down"] = df["Feeling_Down"].map(feeling_down_map)


In [196]:
lack_of_pleasure_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Lack_of_Pleasure"] = df["Lack_of_Pleasure"].astype(str).str.strip()
df["Lack_of_Pleasure"] = df["Lack_of_Pleasure"].map(lack_of_pleasure_map)


In [197]:
feels_pitied_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Feels_Pitied"] = df["Feels_Pitied"].astype(str).str.strip()
df["Feels_Pitied"] = df["Feels_Pitied"].map(feels_pitied_map)


In [198]:
abuse_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2
}

df["Recent_Abuse_Experience"] = df["Recent_Abuse_Experience"].astype(str).str.strip()
df["Recent_Abuse_Experience"] = df["Recent_Abuse_Experience"].map(abuse_map)


In [199]:
fear_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Fear_Something_Bad"] = df["Fear_Something_Bad"].astype(str).str.strip()

df["Fear_Something_Bad"] = df["Fear_Something_Bad"].map(fear_map)


In [200]:
life_feels_hard_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Life_Feels_Hard"] = df["Life_Feels_Hard"].astype(str).str.strip()
df["Life_Feels_Hard"] = df["Life_Feels_Hard"].map(life_feels_hard_map)


In [201]:
restlessness_map = {
    "Rarely or none of the time (less than one day)": 0,
    "Some or a little of the time (1-2 days)": 1,
    "Occasionally or a moderate amount of time (3-4 days)": 2,
    "Most or all of the time (5-7 days)": 3
}

df["Restlessness"] = df["Restlessness"].map(restlessness_map)


In [202]:
difficulty_speaking_map = {
    "Not at all": 0,
    "Several days": 1,
    "More than half the days": 2,
    "Nearly every day": 3
}

df["Difficulty_Speaking_Socially"] = df["Difficulty_Speaking_Socially"].map(difficulty_speaking_map)


In [203]:
low_concentration_map = {
    "Not at all": 0,
    "Several days": 1,
    "More than half the days": 2,
    "Nearly every day": 3
}

df["Low_Concentration"] = df["Low_Concentration"].map(low_concentration_map)


In [204]:
Easy_Fatigue_mapping = {
    "I do not feel weak or fatigued easily.": 0,
    "I somewhat feel weak and fatigued easily, but it doesn't interfere much with my activities.": 1,
    "I often feel weak and fatigued, and it affects my ability to carry out daily activities.": 2,
    "I constantly feel weak and fatigued, making it very difficult to perform even basic tasks.": 3
}
df["Easy_Fatigue"] = df["Easy_Fatigue"].map(Easy_Fatigue_mapping)

In [205]:
difficulty_focus_map = {
    "I can focus just as effectively as usual.": 0,
    "I find it harder to concentrate than usual.": 1,
    "Keeping my mind on one thing is very challenging.": 2,
    "I struggle to concentrate on anything at all.": 3
}

df["Difficulty_Focusing"] = df["Difficulty_Focusing"].map(difficulty_focus_map)


In [206]:
low_appetite_map = {
    "My appetite hasn't changed much.": 0,
    "My appetite isn't what it used to be.": 1,
    "My appetite has significantly worsened.": 2,
    "I have completely lost my appetite.": 3
}

df["Low_Appetite"] = df["Low_Appetite"].map(low_appetite_map)


In [207]:
insomnia_map = {
    "I have no trouble falling asleep, sleep through the night, and wake up feeling refreshed.": 0,
    "I occasionally have difficulty falling asleep, wake up briefly during the night, or feel somewhat tired upon waking.": 1,
    "I often wake up 1-2 times during the night, find it hard to return to sleep, or experience unsatisfying sleep, resulting in fatigue on waking.": 2,
    "I frequently wake up several times during the night, struggle to get back to sleep, and/or experience distressing dreams, nightmares, or night terrors, leaving me exhausted upon waking.": 3
}

df["Insomnia"] = df["Insomnia"].map(insomnia_map)


In [208]:
fatigue_map = {
    "I don't feel weak or fatigued at all.": 0,
    "I feel slightly more weak and fatigued than I used to.": 1,
    "I often feel too weak and fatigued to do much.": 2,
    "I feel too weak and fatigued to do anything.": 3
}

df["Fatigue_Frequency"] = df["Fatigue_Frequency"].map(fatigue_map)


In [209]:
anhedonia_map = {
    "I don't feel joy is absent everywhere.": 0,
    "I worry that joy is fleeting and hard to find.": 1,
    "I sense there are lasting changes in my life that have robbed me of joy.": 2,
    "I believe joy has completely vanished from my life.": 3
}

df["Anhedonia_No_Joy"] = df["Anhedonia_No_Joy"].map(anhedonia_map)


In [210]:

social_withdrawal_map = {
    "I have not lost my desire to be around others.": 0,
    "I am less interested in socializing than I used to be.": 1,
    "I have lost much of my interest in social interactions.": 2,
    "I have completely lost interest in socializing with others.": 3
}

df["Social_Withdrawal"] = df["Social_Withdrawal"].map(social_withdrawal_map)



In [211]:
indecisive_map = {
    "I make choices about as well as I ever have.": 0,
    "I delay making choices more often than I used to.": 1,
    "I find it harder to make decisions compared to before.": 2,
    "I find myself unable to make decisions anymore.": 3
}
df["Indecisiveness"] = df["Indecisiveness"].astype(str).str.strip()
df["Indecisiveness"] = df["Indecisiveness"].map(indecisive_map)




In [212]:
irritability_map = {
    "My temper is rarely irritable.": 0,
    "My temper is more irritable than before.": 1,
    "My temper is irritable by almost anything.": 2,
    "My temper is too irritable to control.": 3
}
df["Irritability"] = df["Irritability"].astype(str).str.strip()
df["Irritability"] = df["Irritability"].map(irritability_map)

In [213]:
high_appetite_map = {
    "Not at all": 0,
    "Several days": 1,
    "More than half the days": 2,
    "Nearly every day": 3
}

df["High_Appetite"] = df["High_Appetite"].astype(str).str.strip()
df["High_Appetite"] = df["High_Appetite"].map(high_appetite_map)


In [214]:
desired_order = [
    "Academic Status",
    "Age",
    "Agitation_Level",
    "Alcohol_Consumption",
    "Anhedonia_No_Joy",
    "Crying_Frequency",
    "Difficulty_Focusing",
    "Difficulty_Speaking_Socially",
    "Easy_Fatigue",
    "Emotional_Alignment_Frequency",
    "Fatigue_Frequency",
    "Fear_Something_Bad",
    "Feeling_Down",
    "Feeling_Insignificant",
    "Feels_Others_Are_Kind",
    "Feels_Pitied",
    "Financial_Pressure",
    "Future_Hopelessness",
    "Gender",
    "Has_Debts",
    "High_Appetite",
    "Hopelessness_EndFeeling",
    "Indecisiveness",
    "Insomnia",
    "Interest_Loss",
    "Irritability",
    "Isolation_Frequency",
    "Lack_of_Pleasure",
    "Life_Feels_Hard",
    "Loneliness_Frequency",
    "Lost_Someone_Recently",
    "Low_Appetite",
    "Low_Concentration",
    "Meaninglessness",
    "Melancholic",
    "No_Support_Frequency",
    "On_Medication",
    "Performance_Decline",
    "Physical_Activity",
    "Presence_Not_Genuine_Frequency",
    "Recent_Abuse_Experience",
    "Relationships_Unimportant_Level",
    "Relationship_Status_Divorced",
    "Relationship_Status_In a Relationship",
    "Relationship_Status_Married",
    "Relationship_Status_Single",
    "Residential_Area_Hall",
    "Residential_Area_Outside Hall",
    "Residential_Area_With family",
    "Restlessness",
    "Satisfied_Living_Environment",
    "Self_Confidence_Erosion",
    "Self_Perceived_Failure",
    "Share_Feelings_Lack",
    "Significant_Ailments",
    "Sleep_Duration",
    "Smoking",
    "Social Economic Status",
    "Social_LeftOut_Level",
    "Social_Media_Hours",
    "Social_Withdrawal",
    "Suicidal_Thoughts",
    "Work_While_Study",
    "Workload_Academic_Demand",

    # Target variables
    "Depression Level (BDI-II) (20 items)",
    "Depression Level (PHQ-9 items)",
    "Depression Level (CES-D) (20 items)",
    "Loneliness Level (UCLA-8 items)"
]

In [215]:
new_df = df.copy()
new_df = new_df.reindex(columns=desired_order)


In [216]:
# new_df.to_csv("/content/drive/MyDrive/Academic_Project/DataSet/dataProcessed.csv", index=False)

In [217]:
for col in new_df.columns:
    print(col)
    print(new_df[col].unique())
    print("------")

Academic Status
[4 3 2 1]
------
Age
[25 23 22 21 24 28 26 20 19 18 27]
------
Agitation_Level
[3 0 2 1]
------
Alcohol_Consumption
[0 1]
------
Anhedonia_No_Joy
[1 2 0 3]
------
Crying_Frequency
[1 3 0 2]
------
Difficulty_Focusing
[0 1 3 2]
------
Difficulty_Speaking_Socially
[1 2 0 3]
------
Easy_Fatigue
[3 0 2 1]
------
Emotional_Alignment_Frequency
[0 3 2 1]
------
Fatigue_Frequency
[3 1 0 2]
------
Fear_Something_Bad
[1 0 2 3]
------
Feeling_Down
[0 1 2 3]
------
Feeling_Insignificant
[0 1 2 3]
------
Feels_Others_Are_Kind
[1 0 2 3]
------
Feels_Pitied
[1 0 2 3]
------
Financial_Pressure
[1 0]
------
Future_Hopelessness
[0 1 2 3]
------
Gender
[1 0]
------
Has_Debts
[0 1]
------
High_Appetite
[1 2 3 0]
------
Hopelessness_EndFeeling
[0 3 1 2]
------
Indecisiveness
[3 2 0 1]
------
Insomnia
[1 0 2 3]
------
Interest_Loss
[1 0 2 3]
------
Irritability
[3 0 2 1]
------
Isolation_Frequency
[0 1 3 2]
------
Lack_of_Pleasure
[1 2 0 3]
------
Life_Feels_Hard
[1 0 3 2]
------
Loneliness_

In [218]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 68 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Academic Status                        500 non-null    int64
 1   Age                                    500 non-null    int64
 2   Agitation_Level                        500 non-null    int64
 3   Alcohol_Consumption                    500 non-null    int64
 4   Anhedonia_No_Joy                       500 non-null    int64
 5   Crying_Frequency                       500 non-null    int64
 6   Difficulty_Focusing                    500 non-null    int64
 7   Difficulty_Speaking_Socially           500 non-null    int64
 8   Easy_Fatigue                           500 non-null    int64
 9   Emotional_Alignment_Frequency          500 non-null    int64
 10  Fatigue_Frequency                      500 non-null    int64
 11  Fear_Something_Bad              

In [219]:
new_df.head()

,Academic Status,Age,Agitation_Level,Alcohol_Consumption,Anhedonia_No_Joy,Crying_Frequency,Difficulty_Focusing,Difficulty_Speaking_Socially,Easy_Fatigue,Emotional_Alignment_Frequency,...,Social_LeftOut_Level,Social_Media_Hours,Social_Withdrawal,Suicidal_Thoughts,Work_While_Study,Workload_Academic_Demand,Depression Level (BDI-II) (20 items),Depression Level (PHQ-9 items),Depression Level (CES-D) (20 items),Loneliness Level (UCLA-8 items)
0,4,25,3,0,1,1,0,1,3,0,...,0,9,0,0,0,1,2,2,1,0
1,3,23,3,1,1,1,1,1,0,3,...,1,9,3,1,0,1,2,1,1,1
2,3,22,3,0,2,3,3,2,2,3,...,1,11,2,1,0,1,3,4,3,2
3,3,21,0,0,0,1,0,0,0,2,...,0,3,1,0,0,1,0,0,0,0
4,3,23,3,0,3,3,3,2,0,0,...,2,3,2,2,0,1,3,4,3,2


In [220]:
clean_df = pd.read_csv("/content/drive/MyDrive/Academic_Project/DataSet/dataProcessed.csv")

In [221]:
for col in clean_df.columns:
    print(col)
    print(clean_df[col].unique())
    print("------")

Gender
[1 0]
------
Relationship_Status_Divorced
[0 1]
------
Relationship_Status_In a Relationship
[0 1]
------
Relationship_Status_Married
[0 1]
------
Relationship_Status_Single
[1 0]
------
Age
[25 23 22 21 24 28 26 20 19 18 27]
------
Academic Status
[4 3 2 1]
------
Work_While_Study
[0 1]
------
Residential_Area_Hall
[1 0]
------
Residential_Area_With family
[0 1]
------
Residential_Area_Outside Hall
[0 1]
------
Social Economic Status
[3 4 2 1 5]
------
Financial_Pressure
[1 0]
------
Has_Debts
[0 1]
------
Satisfied_Living_Environment
[1 0]
------
Lost_Someone_Recently
[0 1]
------
Physical_Activity
[0 1]
------
Significant_Ailments
[0 1]
------
On_Medication
[1 0]
------
Smoking
[0 1]
------
Alcohol_Consumption
[0 1]
------
Sleep_Duration
[6 7 5 4 8 9]
------
Social_Media_Hours
[ 9 11  3  6  1]
------
Workload_Academic_Demand
[1 0]
------
Melancholic
[0 1 2 3]
------
Future_Hopelessness
[0 1 2 3]
------
Self_Perceived_Failure
[1 2 0 3]
------
Interest_Loss
[1 0 2 3]
------
Mea